# OpenAI API — 같은 질문엔 같은 답을 (temperature · seed · 의미 캐시)

`1_1_openai` 에서는 질문을 보내고 답을 받았습니다.
그런데 **같은 질문을 두 번 보내면 답이 매번 조금씩 다릅니다.**
챗봇이 어제와 오늘 다른 정책을 안내한다면 곤란하겠죠.

이 노트북은 답을 **일관되게** 만드는 세 가지 장치를 차례로 붙입니다.

## 이 노트북의 흐름

| 단계 | 내용 |
|---|---|
| **Step 1~2** | 준비 — 라이브러리, API 키, 클라이언트 |
| **Step 3~4** | `temperature` — 답의 '무작위성'을 조절하는 손잡이 |
| **Step 5** | `seed` — 같은 입력이면 같은 답이 나오도록 한 번 더 고정 |
| **Step 6** | 한계 — 질문 **표현**이 달라지면 여전히 답이 달라진다 |
| **Step 7~9** | 의미 캐시 — **비슷한 질문**이면 저장해 둔 같은 답을 돌려준다 |

> **이 노트북에서 기억할 한 가지**
> `temperature=0` 과 `seed` 는 답을 **비슷하게** 만들 뿐 **똑같이** 만들어 주지는 못합니다.
> 같은 질문이든 비슷한 질문이든 **글자 하나 다르지 않은 답**을 원하면,
> **임베딩으로 질문을 비교해 저장된 답을 돌려주는 캐시**가 필요합니다.

## Step 1. 준비

`1_1_openai` 와 같습니다. 키는 `.env` 에서 읽고, 앞 8자만 찍어 확인합니다.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(r"C:\Users\wins\AI_edu\.env", override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


## Step 2. 실험용 질문과 도우미 함수

같은 질문을 여러 번 보내 볼 것이므로 **호출을 함수 하나로** 묶어 둡니다.

- `system_prompt` 에 **"3문장 이내"** 를 넣어 답을 짧게 만듭니다. 비교하기 쉽고 요금도 아낍니다.
- 모델은 `gpt-4.1-mini` 를 씁니다.

> **왜 `gpt-5-nano` 가 아닌가?**
> `gpt-5` 계열 같은 **추론(reasoning) 모델은 `temperature` 를 바꿀 수 없습니다** (기본값 1 고정).
> `temperature` 실험은 `gpt-4.1-mini` 처럼 일반 모델로 해야 합니다.

In [2]:
MODEL = "gpt-4.1-mini"

system_prompt = """
당신은 IT 보안 전문가입니다.
질문에 3문장 이내로 핵심만 답하세요.
"""

def ask(question, temperature=1.0, seed=None):
    """질문 하나를 보내고 답변 글자만 돌려준다."""
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
        temperature=temperature,
        seed=seed,
        max_tokens=200,
    )
    return response.choices[0].message.content

question = "회사에서 가장 먼저 챙겨야 할 보안 대책은 무엇인가요?"

## Step 3. `temperature` 가 높으면 — 매번 다른 답

LLM 은 다음 단어를 **확률로 고릅니다.** `temperature` 는 그 고르는 방식을 정하며, 이 노트북에서는 **0 ~ 1 사이** 값을 씁니다.

| temperature | 고르는 방식 | 어울리는 일 |
|---|---|---|
| `0` | 거의 항상 **가장 확률 높은 단어** | 안내·분류·추출 — 답이 **일정**해야 하는 일 |
| `0.3 ~ 0.7` | 적당히 섞어서 | 일반 대화 |
| `1` (기본값) | 확률 낮은 단어도 자주 | 아이디어·글쓰기 — **다양**해야 하는 일 |

먼저 `temperature=1` 로 **같은 질문을 3번** 보내 봅니다.

In [3]:
for i in range(3):
    print(f"[{i + 1}회차] {ask(question, temperature=1.0)}\n")

[1회차] 최우선은 직원들의 보안 인식 교육과 강력한 비밀번호 정책 수립입니다. 이후 최신 보안 패치 적용과 네트워크 방화벽 설정으로 외부 침입을 차단해야 합니다. 마지막으로 중요 데이터의 백업 및 접근 권한 관리가 필수입니다.

[2회차] 가장 먼저 직원들의 보안 인식 교육을 강화해 사회공학 공격을 방지해야 합니다. 동시에 기본적인 패치 관리와 강력한 비밀번호 정책을 도입해 시스템 취약점을 최소화해야 합니다. 마지막으로, 다단계 인증(MFA)을 적용해 계정 탈취 위험을 줄이는 것이 중요합니다.

[3회차] 가장 먼저 직원들의 보안 인식 교육을 통해 사회공학 공격과 피싱을 예방해야 합니다. 다음으로 강력한 비밀번호 정책과 다중인증(MFA)을 도입해 계정 보안을 강화하세요. 마지막으로 중요 시스템과 데이터를 정기적으로 백업하고 최신 보안 패치를 적용하는 것이 필수입니다.



## Step 4. `temperature=0` — 같은 질문엔 비슷한 답

이번엔 `temperature=0` 으로 같은 질문을 3번 보냅니다.
답이 몇 가지로 나왔는지 `set` 으로 세어 봅니다.

> Step 3 보다 **훨씬 비슷해지지만**, 끝부분 몇 단어가 달라지는 경우가 많습니다.
> `temperature=0` 은 '가장 확률 높은 단어'를 고르라는 뜻일 뿐, 서버 계산의 미세한 차이까지 없애 주지는 못합니다.

In [4]:
answers = [ask(question, temperature=0) for _ in range(3)]

for i, a in enumerate(answers, 1):
    print(f"[{i}회차] {a}\n")

print("서로 다른 답의 개수:", len(set(answers)))

[1회차] 가장 먼저 직원들의 보안 인식 교육을 실시해 사회공학 공격을 예방해야 합니다. 다음으로 강력한 비밀번호 정책과 다중 인증(MFA)을 도입해 계정 탈취를 방지하세요. 마지막으로 최신 보안 패치 적용과 백업 체계를 구축해 시스템 안정성을 확보해야 합니다.

[2회차] 가장 먼저 직원들의 보안 인식 교육을 실시해 사회공학 공격을 예방해야 합니다. 다음으로 강력한 비밀번호 정책과 다중 인증(MFA)을 도입해 계정 탈취를 방지하세요. 마지막으로 최신 보안 패치 적용과 백업 체계를 구축해 시스템 안정성을 확보해야 합니다.

[3회차] 가장 먼저 직원들의 보안 인식 교육을 실시해 사회공학 공격을 예방해야 합니다. 다음으로 강력한 비밀번호 정책과 다중 인증(MFA)을 도입해 계정 탈취를 방지하세요. 마지막으로 최신 보안 패치 적용과 백업 체계를 구축해 시스템 안정성을 확보해야 합니다.

서로 다른 답의 개수: 1


## Step 5. `seed` — 한 번 더 고정

`temperature=0` 이어도 서버 사정(여러 GPU에서 나눠 계산 등)으로 **아주 가끔 한두 단어가 달라질 수 있습니다.**
`seed` 에 아무 정수나 정해서 넣으면, 같은 입력·같은 설정일 때 **최대한 같은 답을 내도록** 요청할 수 있습니다.

```python
ask(question, temperature=0, seed=42)
```

> OpenAI 는 `seed` 를 **"최선을 다해 맞춘다(best effort)"** 고 설명합니다. 100% 보장은 아닙니다.
> 그래서 실무에서는 `temperature=0` + `seed` 를 **함께** 씁니다.
>
> 실행해 보면 **그래도 답이 2~3가지로 갈리는 경우가 흔합니다.**
> 뜻은 거의 같지만 문장 끝이 조금씩 다릅니다. 이것이 모델 설정만으로 할 수 있는 한계입니다.

In [5]:
answers = [ask(question, temperature=0, seed=42) for _ in range(3)]

for i, a in enumerate(answers, 1):
    print(f"[{i}회차] {a}\n")

print("서로 다른 답의 개수:", len(set(answers)))

[1회차] 가장 먼저 직원들의 보안 인식 교육을 실시해 사회공학 공격을 예방해야 합니다. 다음으로 강력한 비밀번호 정책과 다중 인증(MFA)을 도입해 계정 탈취를 방지하세요. 마지막으로 최신 보안 패치 적용과 백업 체계를 구축해 시스템 안정성을 확보해야 합니다.

[2회차] 가장 먼저 직원들의 보안 인식 교육을 실시해 사회공학 공격을 예방해야 합니다. 다음으로 강력한 비밀번호 정책과 다중 인증(MFA)을 도입해 계정 탈취를 방지하세요. 마지막으로 최신 보안 패치 적용과 백업 체계를 구축해 시스템 안정성을 확보해야 합니다.

[3회차] 가장 먼저 직원들의 보안 인식 교육을 실시해 사회공학 공격을 예방해야 합니다. 다음으로 강력한 비밀번호 정책과 다중 인증(MFA)을 도입해 계정 탈취를 방지해야 합니다. 마지막으로 최신 보안 패치와 백신 프로그램을 적용해 시스템 취약점을 최소화해야 합니다.

서로 다른 답의 개수: 2


## Step 6. 한계 — 표현이 다르면 답도 다르다

사용자는 같은 뜻을 **매번 다른 말로** 묻습니다.
뜻이 같은 질문 3개를 `temperature=0`, `seed=42` 로 보내 봅니다.

In [6]:
similar_questions = [
    "회사에서 가장 먼저 챙겨야 할 보안 대책은 무엇인가요?",
    "우리 회사가 제일 먼저 해야 하는 보안 조치가 뭔가요?",
    "기업 보안에서 최우선으로 할 일은 무엇입니까?",
]

answers = [ask(q, temperature=0, seed=42) for q in similar_questions]

for q, a in zip(similar_questions, answers):
    print(f"Q. {q}\nA. {a}\n")

print("서로 다른 답의 개수:", len(set(answers)))

Q. 회사에서 가장 먼저 챙겨야 할 보안 대책은 무엇인가요?
A. 가장 먼저 직원들의 보안 인식 교육을 실시해 사회공학 공격을 예방해야 합니다. 다음으로 강력한 비밀번호 정책과 다중 인증(MFA)을 도입해 계정 탈취를 방지하세요. 마지막으로 최신 보안 패치 적용과 백업 체계를 구축해 시스템 안정성을 확보해야 합니다.

Q. 우리 회사가 제일 먼저 해야 하는 보안 조치가 뭔가요?
A. 가장 먼저 직원들의 보안 인식 교육을 실시해 내부 위협을 줄이세요. 다음으로, 강력한 비밀번호 정책과 다중 인증(MFA)을 도입해 계정 보안을 강화해야 합니다. 마지막으로, 최신 보안 패치와 백신 프로그램을 즉시 적용해 시스템 취약점을 최소화하세요.

Q. 기업 보안에서 최우선으로 할 일은 무엇입니까?
A. 기업 보안의 최우선 과제는 자산 식별 및 위험 평가를 통해 보호 대상과 위협을 명확히 하는 것입니다. 이를 바탕으로 강력한 접근 통제와 지속적인 모니터링 체계를 구축해야 합니다. 마지막으로, 직원 교육과 보안 정책 준수를 통해 내부 취약점을 최소화하는 것이 중요합니다.

서로 다른 답의 개수: 3


**질문 글자가 한 글자라도 다르면, 모델 입장에서는 완전히 다른 입력**입니다.
`temperature` 와 `seed` 는 **같은 입력**을 같게 만들 뿐, 비슷한 입력까지 묶어 주지는 못합니다.

그렇다면 **"이 질문, 전에 받은 질문과 뜻이 같은가?"** 를 먼저 판단하면 됩니다.
여기에 쓰는 것이 **임베딩(embedding)** 입니다.

## Step 7. 임베딩 — 문장의 '뜻'을 숫자로

임베딩은 문장을 **숫자 목록(벡터)** 으로 바꿉니다. **뜻이 비슷한 문장은 비슷한 벡터**가 됩니다.
두 벡터가 얼마나 비슷한지는 **코사인 유사도**(1에 가까울수록 비슷)로 잽니다.

| 쌍 | 기대 |
|---|---|
| 뜻이 같은 질문끼리 | 높다 |
| 주제가 다른 질문 | 낮다 |

실제로 숫자를 찍어 보고, 어디서 **선을 그을지(threshold)** 정합니다.

In [7]:
import numpy as np

EMBED_MODEL = "text-embedding-3-small"

def embed(text):
    """문장 하나를 벡터로 바꾼다. 길이를 1로 맞춰 두면 내적이 곧 코사인 유사도다."""
    v = np.array(openai.embeddings.create(model=EMBED_MODEL, input=text).data[0].embedding)
    return v / np.linalg.norm(v)

base = embed(similar_questions[0])
for q in similar_questions[1:] + ["피싱 메일은 어떻게 구별하나요?", "오늘 점심 메뉴 추천해 줘"]:
    print(f"{float(base @ embed(q)):.3f}  {q}")

0.637  우리 회사가 제일 먼저 해야 하는 보안 조치가 뭔가요?
0.512  기업 보안에서 최우선으로 할 일은 무엇입니까?
0.131  피싱 메일은 어떻게 구별하나요?
0.131  오늘 점심 메뉴 추천해 줘


## Step 8. 의미 캐시 만들기

동작은 단순합니다.

```
새 질문 → 임베딩 → 저장된 질문들과 유사도 비교
            ├─ 가장 비슷한 것이 THRESHOLD 이상  →  저장된 답을 그대로 돌려준다 (API 호출 없음)
            └─ 아니면                           →  모델에 묻고(temperature=0, seed=42), 질문·답을 저장
```

- `THRESHOLD` 는 Step 7 의 숫자를 보고 정합니다. **뜻이 같은 질문은 넘고, 다른 주제는 못 넘는 값**이어야 합니다.
  - 한국어 질문은 뜻이 같아도 유사도가 생각보다 낮게(0.5~0.7) 나옵니다. 여기서는 **0.45** 로 정했습니다.
- 너무 낮으면 **다른 질문에 엉뚱한 답**을, 너무 높으면 **같은 질문인데 새 답**을 줍니다.
- 덤으로, 캐시에서 꺼낸 답은 **API 요금이 들지 않습니다.**

In [8]:
THRESHOLD = 0.45

cache = []   # (질문 벡터, 질문, 답) 목록

def ask_consistent(question):
    """비슷한 질문을 받은 적이 있으면 저장된 답을, 없으면 새로 물어서 저장한 답을 돌려준다."""
    q_vec = embed(question)

    if cache:
        scores = [float(q_vec @ vec) for vec, _, _ in cache]
        best = int(np.argmax(scores))
        if scores[best] >= THRESHOLD:
            _, original, answer = cache[best]
            print(f"   ↳ 캐시 사용 (유사도 {scores[best]:.3f}, 원래 질문: '{original}')")
            return answer

    answer = ask(question, temperature=0, seed=42)
    cache.append((q_vec, question, answer))
    print("   ↳ 새로 질문해서 저장")
    return answer

## Step 9. 비슷한 질문엔 같은 답, 다른 질문엔 새 답

Step 6 의 질문 3개에 **다른 주제 질문 하나**를 섞고, 마지막에 **첫 질문을 한 번 더** 보냅니다.

In [9]:
test_questions = similar_questions + ["피싱 메일은 어떻게 구별하나요?", similar_questions[0]]

results = {}
for q in test_questions:
    print(f"Q. {q}")
    results[q] = ask_consistent(q)
    print(f"A. {results[q]}\n")

print("비슷한 질문 3개의 서로 다른 답 개수:", len({results[q] for q in similar_questions}))

Q. 회사에서 가장 먼저 챙겨야 할 보안 대책은 무엇인가요?
   ↳ 새로 질문해서 저장
A. 가장 먼저 직원들의 보안 인식 교육을 실시해 사회공학 공격을 예방해야 합니다. 다음으로 강력한 비밀번호 정책과 다중 인증(MFA)을 도입해 계정 탈취를 방지하세요. 마지막으로 최신 보안 패치 적용과 백업 체계를 구축해 시스템 안정성을 확보해야 합니다.

Q. 우리 회사가 제일 먼저 해야 하는 보안 조치가 뭔가요?
   ↳ 캐시 사용 (유사도 0.637, 원래 질문: '회사에서 가장 먼저 챙겨야 할 보안 대책은 무엇인가요?')
A. 가장 먼저 직원들의 보안 인식 교육을 실시해 사회공학 공격을 예방해야 합니다. 다음으로 강력한 비밀번호 정책과 다중 인증(MFA)을 도입해 계정 탈취를 방지하세요. 마지막으로 최신 보안 패치 적용과 백업 체계를 구축해 시스템 안정성을 확보해야 합니다.

Q. 기업 보안에서 최우선으로 할 일은 무엇입니까?
   ↳ 캐시 사용 (유사도 0.512, 원래 질문: '회사에서 가장 먼저 챙겨야 할 보안 대책은 무엇인가요?')
A. 가장 먼저 직원들의 보안 인식 교육을 실시해 사회공학 공격을 예방해야 합니다. 다음으로 강력한 비밀번호 정책과 다중 인증(MFA)을 도입해 계정 탈취를 방지하세요. 마지막으로 최신 보안 패치 적용과 백업 체계를 구축해 시스템 안정성을 확보해야 합니다.

Q. 피싱 메일은 어떻게 구별하나요?
   ↳ 새로 질문해서 저장
A. 피싱 메일은 발신자 주소가 의심스럽거나 도메인이 실제 회사와 다릅니다. 문법 오류, 긴급한 요청, 의심스러운 링크나 첨부파일이 포함되어 있는지 확인하세요. 공식 채널로 직접 확인하는 것이 가장 안전합니다.

Q. 회사에서 가장 먼저 챙겨야 할 보안 대책은 무엇인가요?
   ↳ 캐시 사용 (유사도 1.000, 원래 질문: '회사에서 가장 먼저 챙겨야 할 보안 대책은 무엇인가요?')
A. 가장 먼저 직원들의 보안 인식 교육을 실시해 사회공학 공격을 예방해야 합니다. 다음으로 강력한 비밀번호 정책과 다중 인증

---

## 정리

| 장치 | 같게 만드는 범위 | 방법 |
|---|---|---|
| `temperature=0` | **같은 질문** → 비슷한 답 | 가장 확률 높은 단어만 고른다 |
| `seed` | **같은 질문** → 더 비슷한 답 (best effort, 보장 안 됨) | 무작위 시작점을 고정한다 |
| **의미 캐시** | **같은 질문 + 비슷한 질문** → 글자까지 똑같은 답 | 임베딩으로 질문을 비교해 저장된 답을 재사용 |

### 꼭 기억할 것

| | 내용 |
|---|---|
| **추론 모델은 temperature 고정** | `gpt-5` 계열은 `temperature` 를 바꿀 수 없다 |
| **temperature 는 다양성의 손잡이** | 안내·분류는 낮게, 아이디어·글쓰기는 높게 |
| **표현이 다르면 다른 입력** | 비슷한 질문을 묶으려면 임베딩이 필요하다 |
| **THRESHOLD 는 직접 재서 정한다** | 너무 낮으면 엉뚱한 답, 너무 높으면 캐시가 안 걸린다 |
| **캐시는 요금도 줄인다** | 캐시에서 꺼낸 답은 API 호출이 없다 |

### 직접 해 보기

1. Step 3 의 `temperature` 를 `0.5` 로 바꿔 보세요. `1` 일 때와 `0` 일 때의 중간쯤 되나요?
2. `THRESHOLD` 를 `0.5` 로 낮추고 Step 9 를 다시 실행해 보세요. 피싱 질문에 어떤 답이 나오나요?
3. `test_questions` 에 "직원 보안 교육은 꼭 해야 하나요?" 를 넣어 보세요. 캐시가 걸리나요?
4. 지금 캐시는 노트북을 끄면 사라집니다. `cache` 를 파일로 저장하려면 무엇을 저장해야 할까요?